In [1]:
import os

# This forces TensorFlow to use the internal Keras 2 instead of the new Keras 3
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# Natural Language Processing with TensorFlow

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-example-nlp-problems.png)
*A handful of example natural language processing (NLP) and natural language understanding (NLU) problems. These are also often referred to as sequence problems (going from one sequence to another).*

The main goal of [natural language processing (NLP)](https://becominghuman.ai/a-simple-introduction-to-natural-language-processing-ea66a1747b32) is to derive information from natural language.

Natural language is a broad term but you can consider it to cover any of the following:
* Text (such as that contained in an email, blog post, book, Tweet)
* Speech (a conversation you have with a doctor, voice commands you give to a smart speaker)

Under the umbrellas of text and speech there are many different things you might want to do.

If you're building an email application, you might want to scan incoming emails to see if they're spam or not spam (classification).

If you're trying to analyse customer feedback complaints, you might want to discover which section of your business they're for.

> 🔑 **Note:** Both of these types of data are often referred to as *sequences* (a sentence is a sequence of words). So a common term you'll come across in NLP problems is called *seq2seq*, in other words, finding information in one sequence to produce another sequence (e.g. converting a speech command to a sequence of text-based steps).

To get hands-on with NLP in TensorFlow, we're going to practice the steps we've used previously but this time with text data:

```
Text -> turn into numbers -> build a model -> train the model to find patterns -> use patterns (make predictions)
```

> 📖 **Resource:** For a great overview of NLP and the different problems within it, read the article [*A Simple Introduction to Natural Language Processing*](https://becominghuman.ai/a-simple-introduction-to-natural-language-processing-ea66a1747b32).

## What we're going to cover

Let's get specific hey?

* Downloading a text dataset
* Visualizing text data
* Converting text into numbers using tokenization
* Turning our tokenized text into an embedding
* Modelling a text dataset
  * Starting with a baseline (TF-IDF)
  * Building several deep learning text models
    * Dense, LSTM, GRU, Conv1D, Transfer learning
* Comparing the performance of each our models
* Combining our models into an ensemble
* Saving and loading a trained model
* Find the most wrong predictions


## Get helper functions

In past modules, we've created a bunch of helper functions to do small tasks required for our notebooks.

Rather than rewrite all of these, we can import a script and load them in from there.

The script containing our helper functions can be [found on GitHub](https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py).

In [2]:
# Run the below lines if helper functions are not already downloaded:
# Downloading Helper Functions:

#import wget
#url = 'https://raw.githubusercontent.com/Shailya777/Learning-Logs/refs/heads/main/Deep_Learning_with_Tensorflow/helper_functions.py'
#file = wget.download(url)

In [4]:
# Importing helper functions from Downloaded Script:
from helper_functions import unzip_data, plot_loss_curves, compare_historys, create_tensorboard_callback

## Download a text dataset

Let's start by download a text dataset. We'll be using the [Real or Not?](https://www.kaggle.com/c/nlp-getting-started/data) dataset from Kaggle which contains text-based Tweets about natural disasters. 

The Real Tweets are actually about disasters, for example:

```
Jetstar and Virgin forced to cancel Bali flights again because of ash from Mount Raung volcano
```

The Not Real Tweets are Tweets not about disasters (they can be on anything), for example:

```
'Education is the most powerful weapon which you can use to change the world.' Nelson #Mandela #quote
```

For convenience, the dataset has been [downloaded from Kaggle](https://www.kaggle.com/c/nlp-getting-started/data) (doing this requires a Kaggle account) and uploaded as a downloadable zip file. 

Note: The original downloaded data has not been altered to how you would download it from Kaggle.

In [5]:
# Run below lines of code if data is not already downloaded:
# Downloading Text Data:
#url = 'https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip'
#file = wget.download(url)

# Unzipping The Data:
#unzip_data('nlp_getting_started.zip')

Unzipping `nlp_getting_started.zip` gives the following 3 `.csv` files:
* `sample_submission.csv` - an example of the file you'd submit to the Kaggle competition of your model's predictions.
* `train.csv` - training samples of real and not real diaster Tweets.
* `test.csv` - testing samples of real and not real diaster Tweets.

## Visualizing a text dataset

Once you've acquired a new dataset to work with, what should you do first?

Explore it? Inspect it? Verify it? Become one with it?

All correct.

Right now, our text data samples are in the form of `.csv` files. For an easy way to make them visual, let's turn them into pandas DataFrame's.

> 📖 **Reading:** You might come across text datasets in many different formats. Aside from CSV files (what we're working with), you'll probably encounter `.txt` files and `.json` files too. For working with these type of files, I'd recommend reading the two following articles by RealPython:
* [How to Read and Write Files in Python](https://realpython.com/read-write-files-python/)
* [Working with JSON Data in Python](https://realpython.com/python-json/)

In [6]:
# turning .csv Data into Pandas Dataframe:

import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [7]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [8]:
# Shuffling Training Data:
train_df_shuffled = train_df.sample(frac= 1,
                                   random_state= 42)

In [9]:
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


Notice how the training data has a `"target"` column.

We're going to be writing code to find patterns (e.g. different combinations of words) in the `"text"` column of the training dataset to predict the value of the `"target"` column.

The test dataset doesn't have a `"target"` column.

```
Inputs (text column) -> Machine Learning Algorithm -> Outputs (target column)
```

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-text-classification-inputs-and-outputs.png)
*Example text classification inputs and outputs for the problem of classifying whether a Tweet is about a disaster or not.*

In [10]:
# Testing Data:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [11]:
# How many examples of each class are in Training Data:
train_df_shuffled['target'].value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

Since we have two target values, we're dealing with a **binary classification** problem.

It's fairly balanced too, about 60% negative class (`target = 0`) and 40% positive class (`target = 1`).

Where, 

* `1` = a real disaster Tweet
* `0` = not a real disaster Tweet

In [12]:
# Total Number of Samples in Data:
print(f'Training Data Samples: {len(train_df_shuffled)}')
print(f'Test Data Samples: {len(test_df)}')
print(f'Total Number of Samples: {len(train_df_shuffled) + len(test_df)}')

Training Data Samples: 7613
Test Data Samples: 3263
Total Number of Samples: 10876


Alright, seems like we've got a decent amount of training and test data. If anything, we've got an abundance of testing examples, usually a split of 90/10 (90% training, 10% testing) or 80/20 is suffice.

Okay, time to visualize, let's write some code to visualize random text samples.

Question: Why visualize random samples? You could visualize samples in order but this could lead to only seeing a certain subset of data. Better to visualize a substantial quantity (100+) of random samples to get an idea of the different kinds of data you're working with. In machine learning, never underestimate the power of randomness.

In [13]:
# Visualizing 10 Random Training Samples:

import random
random_index = random.randint(0, len(train_df) - 10) # create random indexes not higher than the total number of samples

for row in train_df_shuffled[['text', 'target']][random_index:random_index+10].itertuples():
    _, text, target = row
    print(f'Target: {target}', '(Real Disaster)' if target > 0 else '(Not Real Disaster)')
    print(f'Text:\n {text}\n')

Target: 0 (Not Real Disaster)
Text:
 @David5Fernandez @theblaze trump will get absolutely demolished in a general election.

Target: 0 (Not Real Disaster)
Text:
 BREAKING: Arsenal's Hector Bellerin has been arrested for questioning regarding the disappearance of Eden Hazard.Û_ http://t.co/eCMJ18AzaI

Target: 1 (Real Disaster)
Text:
 A memorial to the millions who perished in the Holodomor has been erected in the U.S. capital. http://t.co/Dj1LWZNIEH http://t.co/I9MxXkzHbL

Target: 1 (Real Disaster)
Text:
 #NowPlaying: Dubstep Hardstyle Trap Messy Mix (event recording) by Alien Casualty on @Mixify http://t.co/m203UL6o7p http://t.co/m203UL6o7p

Target: 1 (Real Disaster)
Text:
 Early wake up call from my sister begging me to come over &amp; ride w/her in the ambulance to the hospital #RODKiai

Target: 1 (Real Disaster)
Text:
 @Dead_Dreamer15 ...because if it were on fire that'd be a safety hazard

Target: 1 (Real Disaster)
Text:
 #Bestnaijamade: 16yr old PKK suicide bomber who detonated b

### Split data into training and validation sets

Since the test set has no labels and we need a way to evalaute our trained models, we'll split off some of the training data and create a validation set.

When our model trains (tries patterns in the Tweet samples), it'll only see data from the training set and we can see how it performs on unseen data using the validation set.

We'll convert our splits from pandas Series datatypes to lists of strings (for the text) and lists of ints (for the labels) for ease of use later.

To split our training dataset and create a validation dataset, we'll use Scikit-Learn's [`train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) method and dedicate 10% of the training samples to the validation set.

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
# Use Train Test Split to Split Training Data into Training and Validation Sets:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled['text'].to_numpy(),
                                                                           train_df_shuffled['target'].to_numpy(),
                                                                           test_size= 0.1,
                                                                           random_state= 42)

In [16]:
# Checking Lengths:
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

In [17]:
# Checking First 10 Examples in training sentences and it's labels:
train_sentences[:10], train_labels[:10]

(array(['@mogacola @zamtriossu i screamed after hitting tweet',
        'Imagine getting flattened by Kurt Zouma',
        '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
        "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
        'Somehow find you and I collide http://t.co/Ee8RpOahPk',
        '@EvaHanderek @MarleyKnysh great times until the bus driver held us hostage in the mall parking lot lmfao',
        'destroy the free fandom honestly',
        'Weapons stolen from National Guard Armory in New Albany still missing #Gunsense http://t.co/lKNU8902JE',
        '@wfaaweather Pete when will the heat wave pass? Is it really going to be mid month? Frisco Boy Scouts have a canoe trip in Okla.',
        'Patient-reported outcomes in long-term survivors of metastatic colorectal cancer - British Journal of Surgery http://t.co/5Yl4DC1Tqt'],
       dtype=object),
 array([0,

## Converting text into numbers

We've got a training set and a validation set containing Tweets and labels.

Our labels are in numerical form (`0` and `1`) but our Tweets are in string form.

Question: What do you think we have to do before we can use a machine learning algorithm with our text data? 

If you answered something along the lines of "turn it into numbers", you're correct. A machine learning algorithm requires its inputs to be in numerical form.

In NLP, there are two main concepts for turning text into numbers:
* **Tokenization** - A straight mapping from word or character or sub-word to a numerical value. There are three main levels of tokenization:
  1. Using **word-level tokenization** with the sentence "I love TensorFlow" might result in "I" being `0`, "love" being `1` and "TensorFlow" being `2`. In this case, every word in a sequence considered a single **token**.
  2. **Character-level tokenization**, such as converting the letters A-Z to values `1-26`. In this case, every character in a sequence considered a single **token**.
  3. **Sub-word tokenization** is in between word-level and character-level tokenization. It involves breaking invidual words into smaller parts and then converting those smaller parts into numbers. For example, "my favourite food is pineapple pizza" might become "my, fav, avour, rite, fo, oo, od, is, pin, ine, app, le, piz, za". After doing this, these sub-words would then be mapped to a numerical value. In this case, every word could be considered multiple **tokens**.
* **Embeddings** - An embedding is a representation of natural language which can be learned. Representation comes in the form of a **feature vector**. For example, the word "dance" could be represented by the 5-dimensional vector `[-0.8547, 0.4559, -0.3332, 0.9877, 0.1112]`. It's important to note here, the size of the feature vector is tuneable. There are two ways to use embeddings: 
  1. **Create your own embedding** - Once your text has been turned into numbers (required for an embedding), you can put them through an embedding layer (such as [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)) and an embedding representation will be learned during model training.
  2. **Reuse a pre-learned embedding** - Many pre-trained embeddings exist online. These pre-trained embeddings have often been learned on large corpuses of text (such as all of Wikipedia) and thus have a good underlying representation of natural language. You can use a pre-trained embedding to initialize your model and fine-tune it to your own specific task.

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-tokenization-vs-embedding.png)
*Example of **tokenization** (straight mapping from word to number) and **embedding** (richer representation of relationships between tokens).*

Question: What level of tokenzation should I use? What embedding should should I choose?

It depends on your problem. You could try character-level tokenization/embeddings and word-level tokenization/embeddings and see which perform best. You might even want to try stacking them (e.g. combining the outputs of your embedding layers using [`tf.keras.layers.concatenate`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/concatenate)). 

If you're looking for pre-trained word embeddings, [Word2vec embeddings](http://jalammar.github.io/illustrated-word2vec/), [GloVe embeddings](https://nlp.stanford.edu/projects/glove/) and many of the options available on [TensorFlow Hub](https://tfhub.dev/s?module-type=text-embedding) are great places to start.

Note: Much like searching for a pre-trained computer vision model, you can search for pre-trained word embeddings to use for your problem. Try searching for something like "use pre-trained word embeddings in TensorFlow".

### Text vectorization (tokenization)


We'll practice tokenzation (mapping our words to numbers) first.

To tokenize our words, we'll use the helpful preprocessing layer [`tf.keras.layers.TextVectorization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization).

The `TextVectorization` layer takes the following parameters:
* `max_tokens` - The maximum number of words in your vocabulary (e.g. 20000 or the number of unique words in your text), includes a value for OOV (out of vocabulary) tokens. 
* `standardize` - Method for standardizing text. Default is `"lower_and_strip_punctuation"` which lowers text and removes all punctuation marks.
* `split` - How to split text, default is `"whitespace"` which splits on spaces.
* `ngrams` - How many words to contain per token split, for example, `ngrams=2` splits tokens into continuous sequences of 2.
* `output_mode` -  How to output tokens, can be `"int"` (integer mapping), `"binary"` (one-hot encoding), `"count"` or `"tf-idf"`. See documentation for more.
* `output_sequence_length` - Length of tokenized sequence to output. For example, if `output_sequence_length=150`, all tokenized sequences will be 150 tokens long.
* `pad_to_max_tokens` - Defaults to `False`, if `True`, the output feature axis will be padded to `max_tokens` even if the number of unique tokens in the vocabulary is less than `max_tokens`. Only valid in certain modes, see docs for more.


In [18]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

In [20]:
# Using Default Parameters for TextVectorization:

text_vectorizer = TextVectorization(max_tokens= None, # how many words in the vocabulary (all of the different words in your text)
                                   standardize= 'lower_and_strip_punctuation', # how to process text
                                   split= 'whitespace', # how to split tokens
                                   ngrams= None, # create groups of n-words?
                                   output_mode= 'int', # how to map tokens to numbers
                                   output_sequence_length= None) # how long should the output sequence of tokens be?
                                   # pad_to_max_tokens= True) # Not valid if using max_tokens=None

We've initialized a `TextVectorization` object with the default settings but let's customize it a little bit for our own use case.

In particular, let's set values for `max_tokens` and `output_sequence_length`.

For `max_tokens` (the number of words in the vocabulary), multiples of 10,000 (`10,000`, `20,000`, `30,000`) or the exact number of unique words in your text (e.g. `32,179`) are common values.

For our use case, we'll use `10,000`.

And for the `output_sequence_length` we'll use the average number of tokens per Tweet in the training set. But first, we'll need to find it.

In [21]:
train_sentences[0]

'@mogacola @zamtriossu i screamed after hitting tweet'

In [22]:
train_sentences[0].split()

['@mogacola', '@zamtriossu', 'i', 'screamed', 'after', 'hitting', 'tweet']

In [23]:
len(train_sentences[0].split())

7

In [24]:
# Average Number of Tokens(Words) in Training Tweets:
round(sum([len(i.split()) for i in train_sentences]) / len(train_sentences))

15

In [25]:
# Setting up TextVectorization with Custom Parameter values:

max_vocab_length = 10000 # max number of words to have in our vocabulary
max_lengths = 15 # max length our sequences will be (e.g. how many words from a Tweet does our model see?)

text_vectorizer = TextVectorization(max_tokens= max_vocab_length,
                                   output_mode= 'int',
                                   output_sequence_length= max_lengths)

To map our `TextVectorization` instance `text_vectorizer` to our data, we can call the `adapt()` method on it whilst passing it our training text.

In [26]:
# Fir the TextVectorizer to Training Data:
text_vectorizer.adapt(train_sentences)

In [27]:
# Creating a Small Sentence and Vectorize it:
sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

It seems we've got a way to turn our text into numbers (in this case, word-level tokenization). Notice the 0's at the end of the returned tensor, this is because we set `output_sequence_length=15`, meaning no matter the size of the sequence we pass to `text_vectorizer`, it always returns a sequence with a length of 15.

How about we try our `text_vectorizer` on a few random sentences?

In [28]:
# Chose a Random Sentence from Training Data and Vectorize it:
random_sentence = random.choice(train_sentences)
print(f'Original Text:\n {random_sentence} \n\n Vectorized Version: {text_vectorizer([random_sentence])}')

Original Text:
 Pakistan air ambulance helicopter crash kills nine http://t.co/8E7rY8eBMf 

 Vectorized Version: [[ 811  307  324  624   85  418 2033    1    0    0    0    0    0    0
     0]]


We can check the unique tokens in our vocabulary using the `get_vocabulary()` method.

In [29]:
# Get the Unique Words in Vocabulary of Text Vectorizer:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]
print(f'Number of Words in Vocabulary: {len(words_in_vocab)}')
print(f'Top 5 Most Common Words: {top_5_words}')
print(f'Bottom 5 Least Common Words: {bottom_5_words}')

Number of Words in Vocabulary: 10000
Top 5 Most Common Words: ['', '[UNK]', 'the', 'a', 'in']
Bottom 5 Least Common Words: ['pages', 'paeds', 'pads', 'padres', 'paddytomlinson1']


### Creating an Embedding using an Embedding Layer

We've got a way to map our text to numbers. How about we go a step further and turn those numbers into an embedding?

The powerful thing about an embedding is it can be learned during training. This means rather than just being static (e.g. `1` = I, `2` = love, `3` = TensorFlow), a word's numeric representation can be improved as a model goes through data samples.

We can see what an embedding of a word looks like by using the [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) layer. 

The main parameters we're concerned about here are:
* `input_dim` - The size of the vocabulary (e.g. `len(text_vectorizer.get_vocabulary()`).
* `output_dim` - The size of the output embedding vector, for example, a value of `100` outputs a  feature vector of size 100 for each word.
* `embeddings_initializer` - How to initialize the embeddings matrix, default is `"uniform"` which randomly initalizes embedding matrix with uniform distribution. This can be changed for using pre-learned embeddings.
* `input_length` - Length of sequences being passed to embedding layer.

In [30]:
from tensorflow.keras.layers import Embedding

In [31]:
tf.random.set_seed(42)

embedding = Embedding(input_dim= max_vocab_length, # set input shape
                     output_dim= 128, # set size of embedding vector
                     embeddings_initializer= 'uniform', # default, intialize randomly
                     name= 'Embedding_1')

In [32]:
embedding

In [33]:
# getting a Random Sentence from Training Sentences to Embed:

random_sentence = random.choice(train_sentences)
print(f'Original Text:\n {random_sentence}')
print(f'Embedded Version:\n')

# Embed The Random Sentence (Turn it into Numerical Representation (into Dense Vectors of Fixed Size)):
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original Text:
 @Stretcher @invalid @Grazed @Rexyy @Towel I see the programme (:
Embedded Version:



<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.01160653,  0.01809665,  0.02675016, ...,  0.0391908 ,
          0.00297855, -0.03284138],
        [ 0.04758129,  0.00689932, -0.03844922, ...,  0.02076887,
          0.02840066, -0.01483346],
        [-0.03835171,  0.02784352, -0.04654069, ...,  0.00886189,
          0.01674709, -0.01165012],
        ...,
        [ 0.00647724,  0.04970315, -0.04525049, ...,  0.02701059,
         -0.01875939,  0.02308153],
        [ 0.00647724,  0.04970315, -0.04525049, ...,  0.02701059,
         -0.01875939,  0.02308153],
        [ 0.00647724,  0.04970315, -0.04525049, ...,  0.02701059,
         -0.01875939,  0.02308153]]], dtype=float32)>

In [34]:
# Checking Out Single Token (word)'s Embedding:
sample_embed[0][0]

<tf.Tensor: shape=(128,), dtype=float32, numpy=
array([ 1.1606526e-02,  1.8096652e-02,  2.6750159e-02, -3.6529861e-02,
        3.9522413e-02, -8.4306709e-03,  8.4008798e-03, -2.6702965e-02,
        3.7757743e-02, -2.2237290e-02, -3.8874090e-02,  4.0448792e-03,
       -1.4621019e-03, -1.1536442e-02,  2.8863553e-02,  2.4004187e-02,
        4.7291029e-02, -6.5696463e-03, -4.4575442e-02,  2.1039221e-02,
       -3.9070107e-02, -4.0440857e-02, -7.1519725e-03, -4.8511304e-02,
        4.4018518e-02,  3.2661919e-02,  2.8190900e-02,  2.0011041e-02,
       -2.3904538e-02,  4.5981240e-02, -4.5442987e-02, -1.2777388e-02,
       -1.5961576e-02, -7.1801320e-03, -2.5081588e-02, -3.0873848e-02,
       -3.9174043e-02, -4.5268595e-02,  4.8297737e-02,  1.5826885e-02,
        6.6598170e-03, -3.7499189e-02,  2.0899478e-02, -4.1685998e-05,
        2.3022804e-02, -4.4507608e-03,  1.6296063e-02, -4.0149260e-02,
        1.9026399e-03,  2.5601033e-02,  3.4857061e-02, -3.0278731e-02,
        4.9499299e-02,  4.131

In [35]:
random_sentence.split()[0]

'@Stretcher'

## Modelling a text dataset

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-inputs-and-outputs-with-shapes-and-models-were-going-to-build.png)
*Once you've got your inputs and outputs prepared, it's a matter of figuring out which machine learning model to build in between them to bridge the gap.*

Now that we've got a way to turn our text data into numbers, we can start to build machine learning models to model it.

To get plenty of practice, we're going to build a series of different models, each as its own experiment. We'll then compare the results of each model and see which one performed best.

More specifically, we'll be building the following:
* **Model 0**: Naive Bayes (baseline)
* **Model 1**: Feed-forward neural network (dense model)
* **Model 2**: LSTM model
* **Model 3**: GRU model
* **Model 4**: Bidirectional-LSTM model
* **Model 5**: 1D Convolutional Neural Network
* **Model 6**: TensorFlow Hub Pretrained Feature Extractor
* **Model 7**: Same as model 6 with 10% of training data

Model 0 is the simplest to acquire a baseline which we'll expect each other of the other deeper models to beat.

Each experiment will go through the following steps:
* Construct the model
* Train the model
* Make predictions with the model
* Track prediction evaluation metrics for later comparison

### Model 0: Getting a baseline

As with all machine learning modelling experiments, it's important to create a baseline model so you've got a benchmark for future experiments to build upon.

To create our baseline, we'll create a Scikit-Learn Pipeline using the TF-IDF (term frequency-inverse document frequency) formula to convert our words to numbers and then model them with the [Multinomial Naive Bayes algorithm](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB). This was chosen via referring to the [Scikit-Learn machine learning map](https://scikit-learn.org/stable/tutorial/machine_learning_map/index.html).

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [37]:
# Creating a Pipeline for Tokenization and Modelling:
model_0 = Pipeline([
    ('TfIdf', TfidfVectorizer()), # convert words to numbers using tfidf
    ('Classifier', MultinomialNB()) # model the text
]) 

In [38]:
# Fitting Training Data and Labels into Pipeline:
model_0.fit(train_sentences, train_labels)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('TfIdf', ...), ('Classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [39]:
# Evaluating The Baseline Model:
baseline_score = model_0.score(val_sentences, val_labels)
baseline_score

0.7926509186351706

In [40]:
# Making Predictions using Baseline Model:
baseline_preds = model_0.predict(val_sentences)
baseline_preds

array([1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0,
       1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1,
       1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,

### Creating an evaluation function for our model experiments

We could evaluate these as they are but since we're going to be evaluating several models in the same way going forward, let's create a helper function which takes an array of predictions and ground truth labels and computes the following:
* Accuracy
* Precision
* Recall
* F1-score

Note: Since we're dealing with a classification problem, the above metrics are the most appropriate. If we were working with a regression problem, other metrics such as MAE (mean absolute error) would be a better choice.

In [41]:
# Creating an Evaluation Function:

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):
    """
      Calculates model accuracy, precision, recall and f1 score of a binary classification model.

      Args:
      -----
      y_true = true labels in the form of a 1D array
      y_pred = predicted labels in the form of a 1D array

      Returns a dictionary of accuracy, precision, recall, f1-score.
    """

    # Calculate Model Accuracy:
    model_accuracy = accuracy_score(y_true= y_true,
                                   y_pred= y_pred) * 100

    # Calculate Model Precision, Recall and F1-Score:
    model_precision, model_recall, model_F1, _ = precision_recall_fscore_support(y_true, y_pred, average= 'weighted')

    model_results = {
        'Accuracy': model_accuracy,
        'Precision': model_precision,
        'Recall': model_recall,
        'F1-Score': model_F1
    }

    return model_results

In [42]:
# Getting Results for Baseline Model:
baseline_results = calculate_results(y_true= val_labels,
                                    y_pred= baseline_preds)
baseline_results

{'Accuracy': 79.26509186351706,
 'Precision': 0.8111390004213173,
 'Recall': 0.7926509186351706,
 'F1-Score': 0.7862189758049549}

### Model 1: A simple dense model

The first "deep" model we're going to build is a single layer dense model. In fact, it's barely going to have a single layer. 

It'll take our text and labels as input, tokenize the text, create an embedding, find the average of the embedding (using Global Average Pooling) and then pass the average through a fully connected layer with one output unit and a sigmoid activation function.

If the previous sentence sounds like a mouthful, it'll make sense when we code it out (remember, if in doubt, code it out).

And since we're going to be building a number of TensorFlow deep learning models, we'll import our `create_tensorboard_callback()` function from `helper_functions.py` to keep track of the results of each. 

In [43]:
from helper_functions import create_tensorboard_callback
from tensorflow.keras.layers import Input, GlobalAveragePooling1D, Dense
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

In [44]:
# Creating a Dictionary to Save Tensorboard Logs:
SAVE_DIR = 'model_logs'

In [45]:
# Building a Model with Functional API:

inputs = Input(shape= (1,), dtype= tf.string) # inputs are 1-dimensional strings
x = text_vectorizer(inputs) # turn the input text into numbers
x = embedding(x) # create an embedding of the numerized numbers
x = GlobalAveragePooling1D()(x) # lower the dimensionality of the embedding
outputs = Dense(units= 1, activation= 'sigmoid')(x) # create the output layer, want binary outputs so use sigmoid activation
model_1 = Model(inputs, outputs, name= 'Model_1_Dense') # construct the model

In [46]:
# Model 1 Summary:
model_1.summary()

Model: "Model_1_Dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_2 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 Embedding_1 (Embedding)     (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [47]:
# Compiling The Model:
model_1.compile(optimizer= Adam(),
               loss= BinaryCrossentropy(),
               metrics= ['accuracy'])

In [48]:
# Fit The Model:
model_1_history = model_1.fit(x= train_sentences,
                             y= train_labels,
                             validation_data= (val_sentences, val_labels),
                             callbacks= [create_tensorboard_callback(dir_name= SAVE_DIR,
                                                                    experiment_name= 'Model_1_Dense')],
                             epochs= 5)

Saving TensorBoard log files to: model_logs/Model_1_Dense/20260213-124533
Epoch 1/5


2026-02-13 12:45:35.888412: I external/local_xla/xla/service/service.cc:163] XLA service 0x73541a1f3390 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-13 12:45:35.888575: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2026-02-13 12:45:35.931458: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-13 12:45:36.029826: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91701
I0000 00:00:1770986736.436611     907 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


215/215 [==============================] - 14s 53ms/step - loss: 0.6096 - accuracy: 0.6920 - val_loss: 0.5355 - val_accuracy: 0.7546
Epoch 2/5
215/215 [==============================] - 11s 50ms/step - loss: 0.4412 - accuracy: 0.8193 - val_loss: 0.4688 - val_accuracy: 0.7861
Epoch 3/5
215/215 [==============================] - -1s -4669us/step - loss: 0.3464 - accuracy: 0.8612 - val_loss: 0.4586 - val_accuracy: 0.7900
Epoch 4/5
215/215 [==============================] - 4s 21ms/step - loss: 0.2850 - accuracy: 0.8915 - val_loss: 0.4636 - val_accuracy: 0.7900
Epoch 5/5
215/215 [==============================] - 5s 21ms/step - loss: 0.2381 - accuracy: 0.9114 - val_loss: 0.4761 - val_accuracy: 0.7874


In [49]:
# Evaluating The Model:
model_1.evaluate(val_sentences, val_labels)

24/24 [==============================] - 0s 16ms/step - loss: 0.4761 - accuracy: 0.7874


[0.4760526716709137, 0.787401556968689]

In [50]:
# Predictions (These come back in the form of Probabilities):
model_1_pred_prob = model_1.predict(val_sentences)
model_1_pred_prob

24/24 [==============================] - -6s -266774us/step


array([[4.10327286e-01],
       [7.46918619e-01],
       [9.97845531e-01],
       [1.09820001e-01],
       [1.10757180e-01],
       [9.35951233e-01],
       [9.10533130e-01],
       [9.92899418e-01],
       [9.68804598e-01],
       [2.65604764e-01],
       [1.19977310e-01],
       [7.06471443e-01],
       [5.35597093e-02],
       [2.20437869e-01],
       [5.10423305e-03],
       [1.61866128e-01],
       [2.66114138e-02],
       [1.02515832e-01],
       [2.40508825e-01],
       [5.52862823e-01],
       [9.11301434e-01],
       [4.27377634e-02],
       [4.49225068e-01],
       [1.03020266e-01],
       [9.57230568e-01],
       [9.98900056e-01],
       [3.38001326e-02],
       [7.10344017e-02],
       [2.70931814e-02],
       [2.17941672e-01],
       [5.70190370e-01],
       [2.40410477e-01],
       [5.18839300e-01],
       [2.04118401e-01],
       [5.27390003e-01],
       [5.71029000e-02],
       [9.94391024e-01],
       [1.42728433e-01],
       [3.54855061e-02],
       [9.98528361e-01],


Since our final layer uses a sigmoid activation function, we get our predictions back in the form of probabilities.

To convert them to prediction classes, we'll use `tf.round()`, meaning prediction probabilities below 0.5 will be rounded to 0 and those above 0.5 will be rounded to 1.

Note: In practice, the output threshold of a sigmoid prediction probability doesn't necessarily have to 0.5. For example, through testing, you may find that a cut off of 0.25 is better for your chosen evaluation metrics. A common example of this threshold cutoff is the [precision-recall tradeoff](https://en.wikipedia.org/wiki/Precision_and_recall#Introduction) (search for the keyword "tradeoff" to learn about the phenomenon).

In [51]:
# Convert Model's Prediction Probabilities to Labels:
model_1_preds = tf.squeeze(tf.round(model_1_pred_prob))

In [52]:
model_1_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [53]:
# Calculating Model_1 results:
model_1_results = calculate_results(y_true= val_labels,
                                   y_pred= model_1_preds)
model_1_results

{'Accuracy': 78.74015748031496,
 'Precision': 0.7914920592553047,
 'Recall': 0.7874015748031497,
 'F1-Score': 0.7846966492209201}

In [54]:
# Is our simple Keras model better than our baseline model?
import numpy as np
np.array(list(model_1_results.values())) > np.array(list(baseline_results.values()))

array([False, False, False, False])

## Visualizing learned embeddings

Our first model (`model_1`) contained an embedding layer (`embedding`) which learned a way of representing words as feature vectors by passing over the training data.

Hearing this for the first few times may sound confusing.

So to further help understand what a text embedding is, let's visualize the embedding our model learned.

In [55]:
# Get the vocabulary from the Text Vectorixation Layer:
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(10000, ['', '[UNK]', 'the', 'a', 'in', 'to', 'of', 'and', 'i', 'is'])

In [56]:
model_1.summary()

Model: "Model_1_Dense"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_2 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 Embedding_1 (Embedding)     (None, 15, 128)           1280000   
                                                                 
 global_average_pooling1d (  (None, 128)               0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1280129 (4.88 MB)
Trainable params: 128

In [57]:
# Get The Weight Matrix of Embedding Layer (these are the numerical patterns between the text in the training dataset the model has learned):

embed_weights = model_1.get_layer('Embedding_1').get_weights()[0]
embed_weights

array([[-0.01126247,  0.03992427, -0.0277004 , ...,  0.0116006 ,
        -0.03641381,  0.00519965],
       [-0.00776336,  0.01408793,  0.03573605, ...,  0.02977233,
         0.01668786,  0.02536205],
       [ 0.00416566,  0.02538992,  0.02556075, ..., -0.05481832,
        -0.04007016,  0.02224005],
       ...,
       [ 0.03554762, -0.04314221,  0.0078249 , ..., -0.01197311,
         0.0060445 , -0.01150788],
       [-0.06433646,  0.00123309,  0.01867046, ..., -0.00924701,
        -0.0780734 , -0.03930804],
       [-0.10168792, -0.04715617,  0.0884571 , ..., -0.06124551,
        -0.06426581, -0.10371792]], dtype=float32)

In [58]:
# Same size as vocab size and embedding_dim (each word is a embedding_dim size vector):
embed_weights.shape

(10000, 128)

Now we've got these two objects, we can use the [Embedding Projector tool](http://projector.tensorflow.org/_) to visualize our embedding. 

To use the Embedding Projector tool, we need two files:
* The embedding vectors (same as embedding weights).
* The meta data of the embedding vectors (the words they represent - our vocabulary).

Right now, we've got of these files as Python objects. To download them to file, we're going to [use the code example available on the TensorFlow word embeddings tutorial page](https://www.tensorflow.org/tutorials/text/word_embeddings#retrieve_the_trained_word_embeddings_and_save_them_to_disk).

In [59]:
# Code below is adapted from: https://www.tensorflow.org/tutorials/text/word_embeddings#retrieve_the_trained_word_embeddings_and_save_them_to_disk
import io

# Create output writers
out_v = io.open("embedding_vectors.tsv", "w", encoding="utf-8")
out_m = io.open("embedding_metadata.tsv", "w", encoding="utf-8")

# Write embedding vectors and words to file
for num, word in enumerate(words_in_vocab):
   if num == 0: 
      continue # skip padding token
   vec = embed_weights[num]
   out_m.write(word + "\n") # write words to file
   out_v.write("\t".join([str(x) for x in vec]) + "\n") # write corresponding word vector to file
out_v.close()
out_m.close()

Once you've downloaded the embedding vectors and metadata, you can visualize them using Embedding Vector tool:
1. Go to  http://projector.tensorflow.org/
2. Click on "Load data"
3. Upload the two files you downloaded (`embedding_vectors.tsv` and `embedding_metadata.tsv`)
4. Explore
5. Optional: You can share the data you've created by clicking "Publish"

Are words with similar meanings close together?

Remember, they might not be. The embeddings we downloaded are how our model interprets words, not necessarily how we interpret them. 

Also, since the embedding has been learned purely from Tweets, it may contain some strange values as Tweets are a very unique style of natural language.

> 🤔 **Question:** Do you have to visualize embeddings every time?

No. Although helpful for gaining an intuition of what natural language embeddings are, it's not completely necessary. Especially as the dimensions of your vocabulary and embeddings grow, trying to comprehend them would become an increasingly difficult task.

## Recurrent Neural Networks (RNN's)

For our next series of modelling experiments we're going to be using a special kind of neural network called a **Recurrent Neural Network (RNN)**.

The premise of an RNN is simple: use information from the past to help you with the future (this is where the term recurrent comes from). In other words, take an input (`X`) and compute an output (`y`) based on all previous inputs.

This concept is especially helpful when dealing with sequences such as passages of natural language text (such as our Tweets).

For example, when you read this sentence, you take into context the previous words when deciphering the meaning of the current word dog. 

See what happened there? 

I put the word "dog" at the end which is a valid word but it doesn't make sense in the context of the rest of the sentence.

When an RNN looks at a sequence of text (already in numerical form), the patterns it learns are continually updated based on the order of the sequence. 

For a simple example, take two sentences:
1. Massive earthquake last week, no?
2. No massive earthquake last week.

Both contain exactly the same words but have different meaning. The order of the words determines the meaning (one could argue punctuation marks also dictate the meaning but for simplicity sake, let's stay focused on the words).

Recurrent neural networks can be used for a number of sequence-based problems:
* **One to one:** one input, one output, such as image classification.
* **One to many:** one input, many outputs, such as image captioning (image input, a sequence of text as caption output).
* **Many to one:** many inputs, one outputs, such as text classification (classifying a Tweet as real diaster or not real diaster).
* **Many to many:** many inputs, many outputs, such as machine translation (translating English to Spanish) or speech to text (audio wave as input, text as output).

When you come across RNN's in the wild, you'll most likely come across variants of the following:
* Long short-term memory cells (LSTMs).
* Gated recurrent units (GRUs).
* Bidirectional RNN's (passes forward and backward along a sequence, left to right and right to left).

For a deeper understanding of what's happening behind the scenes of the code we're about to write, I'd recommend the following resources:

> * [MIT Deep Learning Lecture on Recurrent Neural Networks](https://youtu.be/SEnXr6v2ifU) - explains the background of recurrent neural networks and introduces LSTMs.
> * [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/) by Andrej Karpathy - demonstrates the power of RNN's with examples generating various sequences.
> * [Understanding LSTMs](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) by Chris Olah - an in-depth (and technical) look at the mechanics of the LSTM cell, possibly the most popular RNN building block.

### Model 2: LSTM

We're going to start with an LSTM-powered RNN.

To harness the power of the LSTM cell (LSTM cell and LSTM layer are often used interchangably) in TensorFlow, we'll use [`tensorflow.keras.layers.LSTM()`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM).

![](https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/08-RNN-architecture-coloured-block-edition.png)
*Coloured block example of the structure of an recurrent neural network.*

Our model is going to take on a very similar structure to `model_1`:

```
Input (text) -> Tokenize -> Embedding -> Layers -> Output (label probability)
```

The main difference will be that we're going to add an LSTM layer between our embedding and output.

And to make sure we're not getting reusing trained embeddings (this would involve data leakage between models, leading to an uneven comparison later on), we'll create another embedding layer (`model_2_embedding`) for our model. The `text_vectorizer` layer can be reused since it doesn't get updated during training.

> 🔑 **Note:** The reason we use a new embedding layer for each model is since the embedding layer is a *learned* representation of words (as numbers), if we were to use the same embedding layer (`embedding_1`) for each model, we'd be mixing what one model learned with the next. And because we want to compare our models later on, starting them with their own embedding layer each time is a better idea.

In [60]:
from tensorflow.keras.layers import LSTM

In [61]:
# Creating Embedding Layer for Model (new embedding layer for each model):
tf.random.set_seed(42)
model_2_embedding = Embedding(input_dim= max_vocab_length,
                             output_dim= 128,
                             embeddings_initializer= 'uniform',
                             name= 'Embedding_2')

In [62]:
# Model_2:

inputs = Input(shape= (1,), dtype= tf.string)
x = text_vectorizer(inputs)
x = model_2_embedding(x)
# print(x.shape)
# x = LSTM(units= 64, return_sequences= True)(x) # return vector for each word in the Tweet (you can stack RNN cells as long as return_sequences=True)
# print(x.shape)
x= LSTM(units= 64)(x) # return vector for whole sequence
# print(x.shape)
# x = Dense(units= 64, activation= 'relu')(x) # optional dense layer on top of output of LSTM cell
outputs = Dense(units= 1, activation= 'sigmoid')(x)
model_2 = Model(inputs, outputs, name= 'Model_2_LSTM')

Note: Reading the documentation for the [TensorFlow LSTM layer](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM), you'll find a plethora of parameters. Many of these have been tuned to make sure they compute as fast as possible. The main ones you'll be looking to adjust are `units` (number of hidden units) and `return_sequences` (set this to `True` when stacking LSTM or other recurrent layers).

Now we've got our LSTM model built, let's compile it using `"binary_crossentropy"` loss and the Adam optimizer.

In [63]:
# Model Summary:
model_2.summary()

Model: "Model_2_LSTM"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_2 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 Embedding_2 (Embedding)     (None, 15, 128)           1280000   
                                                                 
 lstm (LSTM)                 (None, 64)                49408     
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1329473 (5.07 MB)
Trainable params: 1329473 (5.07 MB)
Non-trainable params: 0 (0.00 Byte)
________________

If you'd like to know where this number comes from, I recommend going through the above resources as well the following on calculating the number of parameters in an LSTM cell:
* [Stack Overflow answer to calculate the number of parameters in an LSTM cell](https://stackoverflow.com/questions/38080035/how-to-calculate-the-number-of-parameters-of-an-lstm-network) by Marcin Możejko
* [Calculating number of parameters in a LSTM unit and layer](https://medium.com/@priyadarshi.cse/calculating-number-of-parameters-in-a-lstm-unit-layer-7e491978e1e4) by Shridhar Priyadarshi

In [64]:
# Compiling The Model:
model_2.compile(optimizer= Adam(),
               loss= BinaryCrossentropy(),
               metrics= ['accuracy'])

In [65]:
# Fit The Model:
model_2_history = model_2.fit(x= train_sentences,
                             y= train_labels,
                             epochs= 5,
                             validation_data= (val_sentences, val_labels),
                             callbacks= [create_tensorboard_callback(dir_name= SAVE_DIR,
                                                                    experiment_name= 'Model_2_LSTM')])

Saving TensorBoard log files to: model_logs/Model_2_LSTM/20260213-124717
Epoch 1/5
215/215 [==============================] - 17s 70ms/step - loss: 0.5092 - accuracy: 0.7459 - val_loss: 0.4530 - val_accuracy: 0.7795
Epoch 2/5
215/215 [==============================] - 8s 37ms/step - loss: 0.3178 - accuracy: 0.8711 - val_loss: 0.5062 - val_accuracy: 0.7769
Epoch 3/5
215/215 [==============================] - 7s 31ms/step - loss: 0.2211 - accuracy: 0.9161 - val_loss: 0.5690 - val_accuracy: 0.7625
Epoch 4/5
215/215 [==============================] - 6s 28ms/step - loss: 0.1601 - accuracy: 0.9441 - val_loss: 0.5965 - val_accuracy: 0.7769
Epoch 5/5
215/215 [==============================] - 5s 24ms/step - loss: 0.1136 - accuracy: 0.9585 - val_loss: 0.8130 - val_accuracy: 0.7625


In [66]:
# Making Predictions Using Model 2 on Validation Data:
model_2_pred_prob = model_2.predict(val_sentences)
model_2_pred_prob[:10]

24/24 [==============================] - 1s 12ms/step


array([[0.02362596],
       [0.71754074],
       [0.9993606 ],
       [0.06534005],
       [0.00618958],
       [0.99943835],
       [0.78837365],
       [0.99975866],
       [0.9996259 ],
       [0.2907081 ]], dtype=float32)

In [67]:
# Converting Model's Prediction Probabilites to Labels:
model_2_preds = tf.squeeze(tf.round(model_2_pred_prob))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [68]:
# Calculating Model 2 Results:
model_2_results = calculate_results(y_true= val_labels,
                                   y_pred= model_2_preds)
model_2_results

{'Accuracy': 76.24671916010499,
 'Precision': 0.7631352147335341,
 'Recall': 0.7624671916010499,
 'F1-Score': 0.7607595644846901}

### Model 3: GRU

Another popular and effective RNN component is the GRU or gated recurrent unit.

The GRU cell has similar features to an LSTM cell but has less parameters.

Resource: resources to learn more:
* [Gated Recurrent Unit](https://en.wikipedia.org/wiki/Gated_recurrent_unit) Wikipedia page
* [Understanding GRU networks](https://towardsdatascience.com/understanding-gru-networks-2ef37df6c9be) by Simeon Kostadinov

To use the GRU cell in TensorFlow, we can call the [`tensorflow.keras.layers.GRU()`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GRU) class.

The architecture of the GRU-powered model will follow the same structure we've been using:

```
Input (text) -> Tokenize -> Embedding -> Layers -> Output (label probability)
```

Again, the only difference will be the layer(s) we use between the embedding and the output.

In [69]:
from tensorflow.keras.layers import GRU

In [70]:
# Creating Embedding Layer for Model (new embedding layer for each model):
model_3_embedding = Embedding(input_dim= max_vocab_length,
                             output_dim= 128,
                             embeddings_initializer= 'uniform',
                             name= 'Embedding_3')

In [74]:
# Model_3:

tf.random.set_seed(42)
inputs = Input(shape= (1,), dtype= tf.string)
x = text_vectorizer(inputs)
x = model_3_embedding(x)
# print(x.shape)
x = GRU(units= 64)(x) # stacking recurrent cells requires return_sequences=True
# print(x.shape)
# x = LSTM(units= 64, return_sequences= True)(x)
# print(x.shape)
# x = GRU(units= 64)(x)
# print(x.shape)
# x = Dense(units= 64, activation= 'relu')(x)
outputs = Dense(units= 1, activation= 'sigmoid')(x)

model_3 = Model(inputs, outputs, name= 'Model_3_GRU')

In [75]:
# Model Summary:
model_3.summary()

Model: "Model_3_GRU"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_6 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_2 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 Embedding_3 (Embedding)     (None, 15, 128)           1280000   
                                                                 
 gru_4 (GRU)                 (None, 64)                37248     
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 1317313 (5.03 MB)
Trainable params: 1317313 (5.03 MB)
Non-trainable params: 0 (0.00 Byte)
_________________

In [76]:
# Compiling The Model:
model_3.compile(optimizer= Adam(),
               loss= BinaryCrossentropy(),
               metrics= ['accuracy'])

In [77]:
# Fitting Model 3:
model_3_history = model_3.fit(x= train_sentences,
                             y= train_labels,
                             validation_data= (val_sentences, val_labels),
                             epochs= 5,
                             callbacks= [create_tensorboard_callback(dir_name= SAVE_DIR,
                                                                    experiment_name= 'Model_3_GRU')])

Saving TensorBoard log files to: model_logs/Model_3_GRU/20260213-125727
Epoch 1/5
215/215 [==============================] - 17s 70ms/step - loss: 0.5311 - accuracy: 0.7224 - val_loss: 0.4549 - val_accuracy: 0.7769
Epoch 2/5
215/215 [==============================] - 8s 35ms/step - loss: 0.3206 - accuracy: 0.8680 - val_loss: 0.4888 - val_accuracy: 0.7835
Epoch 3/5
215/215 [==============================] - 7s 31ms/step - loss: 0.2175 - accuracy: 0.9180 - val_loss: 0.5619 - val_accuracy: 0.7690
Epoch 4/5
215/215 [==============================] - 7s 31ms/step - loss: 0.1542 - accuracy: 0.9463 - val_loss: 0.6210 - val_accuracy: 0.7822
Epoch 5/5
215/215 [==============================] - 7s 32ms/step - loss: 0.1158 - accuracy: 0.9610 - val_loss: 0.5847 - val_accuracy: 0.7808


In [78]:
# Making Predictions Using Model_3 on Validation Data:
model_3_pred_prob = model_3.predict(val_sentences)
model_3_pred_prob[:10]

24/24 [==============================] - 0s 11ms/step


array([[0.30047652],
       [0.93184286],
       [0.9966007 ],
       [0.16591023],
       [0.01348646],
       [0.98646134],
       [0.7672279 ],
       [0.99572587],
       [0.9930137 ],
       [0.48697352]], dtype=float32)

In [79]:
# Converting Model's Prediction probabilities to Labels:
model_3_preds = tf.squeeze(tf.round(model_3_pred_prob))
model_3_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [80]:
# Calculating Model 3 Results:
model_3_results = calculate_results(y_true= val_labels,
                                   y_pred= model_3_preds)
model_3_results

{'Accuracy': 78.08398950131233,
 'Precision': 0.781027063798433,
 'Recall': 0.7808398950131233,
 'F1-Score': 0.7797836213601502}

### Model 4: Bidirectonal RNN model 

Now we're going to look into another kind of RNN, the bidirectional RNN.

A standard RNN will process a sequence from left to right, where as a bidirectional RNN will process the sequence from left to right and then again from right to left.

Intuitively, this can be thought of as if you were reading a sentence for the first time in the normal fashion (left to right) but for some reason it didn't make sense so you traverse back through the words and go back over them again (right to left).

In practice, many sequence models often see and improvement in performance when using bidirectional RNN's.

However, this improvement in performance often comes at the cost of longer training times and increased model parameters (since the model goes left to right and right to left, the number of trainable parameters doubles).

Once again, TensorFlow helps us out by providing the [`tensorflow.keras.layers.Bidirectional`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional) class. We can use the `Bidirectional` class to wrap our existing RNNs, instantly making them bidirectional.

In [81]:
from tensorflow.keras.layers import Bidirectional

In [82]:
# Creating Embedding Layer for Model (new embedding layer for each model):
model_4_embedding = Embedding(input_dim= max_vocab_length,
                             output_dim= 128,
                             embeddings_initializer= 'uniform',
                             name= 'Embedding_4')

In [83]:
# Model_4:

tf.random.set_seed(42)

inputs = Input(shape= (1,), dtype= tf.string)
x = text_vectorizer(inputs)
x = model_4_embedding(x)
x = Bidirectional(LSTM(units= 64))(x)
outputs = Dense(units= 1, activation= 'sigmoid')(x)

model_4 = Model(inputs, outputs, name= 'Model_4_BiDirectional')

In [84]:
# Model Summary:
model_4.summary()

Model: "Model_4_BiDirectional"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_7 (InputLayer)        [(None, 1)]               0         
                                                                 
 text_vectorization_2 (Text  (None, 15)                0         
 Vectorization)                                                  
                                                                 
 Embedding_4 (Embedding)     (None, 15, 128)           1280000   
                                                                 
 bidirectional (Bidirection  (None, 128)               98816     
 al)                                                             
                                                                 
 dense_6 (Dense)             (None, 1)                 129       
                                                                 
Total params: 1378945 (5.26 MB)
Trainable par

You can use the `Bidirectional` wrapper on any RNN cell in TensorFlow. For example, `layers.Bidirectional(layers.GRU(64))` creates a bidirectional GRU cell.

In [86]:
# Compiling The Model:
model_4.compile(optimizer= Adam(),
               loss= BinaryCrossentropy(),
               metrics= ['accuracy'])

In [88]:
# Fitting The Model:
model_4_history = model_4.fit(x= train_sentences,
                             y= train_labels,
                             validation_data= (val_sentences, val_labels),
                             epochs= 5,
                             callbacks= [create_tensorboard_callback(dir_name= SAVE_DIR,
                                                                    experiment_name= 'Model_4_BiDirectional')])

Saving TensorBoard log files to: model_logs/Model_4_BiDirectional/20260213-130805
Epoch 1/5
215/215 [==============================] - 18s 37ms/step - loss: 0.5095 - accuracy: 0.7457 - val_loss: 0.4608 - val_accuracy: 0.7782
Epoch 2/5
215/215 [==============================] - 6s 29ms/step - loss: 0.3122 - accuracy: 0.8723 - val_loss: 0.5311 - val_accuracy: 0.7690
Epoch 3/5
215/215 [==============================] - 7s 32ms/step - loss: 0.2095 - accuracy: 0.9210 - val_loss: 0.5824 - val_accuracy: 0.7664
Epoch 4/5
215/215 [==============================] - 7s 31ms/step - loss: 0.1415 - accuracy: 0.9539 - val_loss: 0.6695 - val_accuracy: 0.7717
Epoch 5/5
215/215 [==============================] - 6s 27ms/step - loss: 0.0995 - accuracy: 0.9660 - val_loss: 0.7437 - val_accuracy: 0.7612


In [89]:
# making Predictions Using Model_4 on Validation Data:
model_4_pred_prob = model_4.predict(val_sentences)
model_4_pred_prob[:10]

24/24 [==============================] - 1s 15ms/step


array([[0.03936329],
       [0.922308  ],
       [0.99970764],
       [0.12514138],
       [0.00542299],
       [0.99829024],
       [0.9928853 ],
       [0.99981624],
       [0.99978703],
       [0.11375186]], dtype=float32)

In [90]:
# Converting Model's Prediction Probabilites to Labels:
model_4_preds = tf.squeeze(tf.round(model_4_pred_prob))
model_4_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [91]:
# Calculating Model 4 Results:
model_4_results = calculate_results(y_true= val_labels,
                                   y_pred= model_4_preds)
model_4_results

{'Accuracy': 76.11548556430446,
 'Precision': 0.760954485432404,
 'Recall': 0.7611548556430446,
 'F1-Score': 0.7601563628867408}

## Convolutional Neural Networks for Text

You might've used convolutional neural networks (CNNs) for images before but they can also be used for sequences.

The main difference between using CNNs for images and sequences is the shape of the data. Images come in 2-dimensions (height x width) where as sequences are often 1-dimensional (a string of text).

So to use CNNs with sequences, we use a 1-dimensional convolution instead of a 2-dimensional convolution.

A typical CNN architecture for sequences will look like the following: 

```
Inputs (text) -> Tokenization -> Embedding -> Layers -> Outputs (class probabilities)
```

You might be thinking "that just looks like the architecture layout we've been using for the other models..."

And you'd be right.

The difference again is in the layers component. Instead of using an LSTM or GRU cell, we're going to use a [`tensorflow.keras.layers.Conv1D()`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv1D) layer followed by a [`tensorflow.keras.layers.GlobablMaxPool1D()`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/GlobalMaxPool1D) layer.

> 📖 **Resource:** The intuition here is explained succinctly in the paper [*Understanding Convolutional Neural Networks for Text Classification*](https://www.aclweb.org/anthology/W18-5408.pdf), where they state that CNNs classify text through the following steps:
1. 1-dimensional convolving filters are used as ngram detectors, each filter specializing in a closely-related family of ngrams (an ngram is a collection of n-words, for example, an ngram of 5 might result in "hello, my name is Daniel").
2. Max-pooling over time extracts the relevant ngrams for making a decision.
3. The rest of the network classifies the text based on this information.

### Model 5: Conv1D

Before we build a full 1-dimensional CNN model, let's see a 1-dimensional convolutional layer (also called a **temporal convolution**) in action.

We'll first create an embedding of a sample of text and experiment passing it through a `Conv1D()` layer and `GlobalMaxPool1D()` layer.

In [92]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPool1D

In [93]:
# Testing of Embedding, 1D Convolution Layer and Global Max Pool 1D Layer:

# Turn Target Sentence into Embedding:
embedding_test = embedding(text_vectorizer(['this is a test sentence']))

# Convolve over Traget Sequence 5 words at a Time:
conv_1d = Conv1D(filters= 32, kernel_size= 5, activation= 'relu')

# Pass Embedding through 1D Convolutional Layer:
conv_1d_output = conv_1d(embedding_test)

# Get The Most Important features:
max_pool = GlobalMaxPool1D()
max_pool_output = max_pool(conv_1d_output)

embedding_test.shape, conv_1d_output.shape, max_pool_output.shape

(TensorShape([1, 15, 128]), TensorShape([1, 11, 32]), TensorShape([1, 32]))

Notice the output shapes of each layer.

The embedding has an output shape dimension of the parameters we set it to (`input_length=15` and `output_dim=128`).

The 1-dimensional convolutional layer has an output which has been compressed inline with its parameters. And the same goes for the max pooling layer output.

Our text starts out as a string but gets converted to a feature vector of length 64 through various transformation steps (from tokenization to embedding to 1-dimensional convolution to max pool).

In [95]:
# outputs of Each layer:
embedding_test[:1], conv_1d_output[:1], max_pool_output[:1]

(<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
 array([[[ 0.04345594,  0.05595118, -0.05119548, ..., -0.01207633,
          -0.03786957,  0.0194117 ],
         [-0.06850044, -0.08254688,  0.0470296 , ..., -0.01713939,
          -0.07301797, -0.0673786 ],
         [-0.01713163,  0.01299601,  0.00547206, ..., -0.00951105,
           0.00134932,  0.01054939],
         ...,
         [-0.01126247,  0.03992427, -0.0277004 , ...,  0.0116006 ,
          -0.03641381,  0.00519965],
         [-0.01126247,  0.03992427, -0.0277004 , ...,  0.0116006 ,
          -0.03641381,  0.00519965],
         [-0.01126247,  0.03992427, -0.0277004 , ...,  0.0116006 ,
          -0.03641381,  0.00519965]]], dtype=float32)>,
 <tf.Tensor: shape=(1, 11, 32), dtype=float32, numpy=
 array([[[3.38876154e-03, 2.92227603e-04, 0.00000000e+00, 0.00000000e+00,
          2.04475038e-02, 3.31998132e-02, 0.00000000e+00, 0.00000000e+00,
          0.00000000e+00, 0.00000000e+00, 1.67407855e-01, 3.36940102e-02,
          0.